# 04 · Model benchmark: same tasks, same data, many models

Every model is scored on the same labeled data, so upgrades are judged by numbers.

| Dataset | Task | Gold |
|---|---|---|
| `cardiffnlp/tweet_sentiment_multilingual` (es + en test) | `MESSAGE_SENTIMENT` | human labels |
| Synthetic Spanish customer-service chats | `SENTIMENT_ARC` (initial / final) | requested arc (by construction) |

Metrics: accuracy, macro-F1, Cohen's κ, coverage (share of items answered without error), p50 latency, cost per 1k items.
Results are cached per (task, model, input) under `data/processed/cache/`, so re-running only pays for new cells.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import os

import pandas as pd
from dotenv import load_dotenv

from genaianalysis.utils.paths import data_processed_dir, project_dir, reports_dir

load_dotenv(project_dir(".env"))
os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")
logging.basicConfig(level=logging.ERROR)  # WARNING shows every retry
pd.set_option("display.max_colwidth", 120)

In [ ]:
from genaianalysis.eval import score
from genaianalysis.ingest.hf import tweet_sentiment
from genaianalysis.ingest.synthetic import generate_chats
from genaianalysis.run import run_task
from genaianalysis.schema import load_jsonl, save_jsonl
from genaianalysis.tasks.sentiment import MESSAGE_SENTIMENT, SENTIMENT_ARC

## Config
Models are pydantic-ai strings, so adding a provider or a new model is one line. Only models whose key is in `.env` run.
`RPM` paces requests below the Gemini free-tier quota (15 requests/min per model); raise it on a paid tier.

In [ ]:
CANDIDATES = {
    "google:gemini-flash-lite-latest": "GEMINI_API_KEY",
    "google:gemini-flash-latest": "GEMINI_API_KEY",
    "anthropic:claude-haiku-4-5": "ANTHROPIC_API_KEY",
    "anthropic:claude-sonnet-5": "ANTHROPIC_API_KEY",
}
MODELS = [m for m, key in CANDIDATES.items() if os.getenv(key) or os.getenv("GOOGLE_API_KEY") and key == "GEMINI_API_KEY"]
N_TWEETS = 100  # per language
N_CHATS = 45    # 5 per (initial, final) arc
RPM = 12
MODELS

## 1 · Tweets (real, human-labeled)

In [ ]:
tweets = {}
for lang in ["spanish", "english"]:
    path = data_processed_dir(f"tweets_{lang}_test_{N_TWEETS}.jsonl")
    if not path.exists():
        save_jsonl(tweet_sentiment(lang, n=N_TWEETS), path)
    tweets[lang] = load_jsonl(path)

pd.Series([c.labels["sentiment"] for c in tweets["spanish"]]).value_counts()

In [ ]:
tweet_results = {
    lang: pd.concat([await run_task(MESSAGE_SENTIMENT, convs, m, rpm=RPM) for m in MODELS])
    for lang, convs in tweets.items()
}
tweet_scores = pd.concat(
    [score(r, tweets[lang], "sentiment").assign(language=lang) for lang, r in tweet_results.items()]
)
tweet_scores

Errors, if any (quota, safety blocks, schema failures):

In [ ]:
pd.concat(tweet_results.values()).dropna(subset=["error"])[["model", "id", "error"]]

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best = tweet_scores[tweet_scores.language == "spanish"].iloc[0]["model"]
r = tweet_results["spanish"].query("model == @best and error.isna()")
gold = {c.id: c.labels["sentiment"] for c in tweets["spanish"]}
ConfusionMatrixDisplay.from_predictions(r["id"].map(gold), r["sentiment"], normalize="true", values_format=".2f").ax_.set_title(best);

Misclassified examples of the best model, to read what "wrong" looks like (sarcasm? label noise?):

In [ ]:
text = {c.id: c.messages[0].text for c in tweets["spanish"]}
wrong = r[r["id"].map(gold) != r["sentiment"]]
wrong.assign(text=wrong["id"].map(text), gold=wrong["id"].map(gold))[["text", "gold", "sentiment"]].head(10)

## 2 · Synthetic customer-service chats (the legacy WhatsApp use case)

Gold = the arc the generator was asked for. Prefer a generator from a different model family than the judges, and read a few chats: a generator that ignores the brief creates label noise.

In [ ]:
# free-tier gemini-flash-latest allows only 20 requests/day, so Flash-Lite is the fallback generator
GENERATOR = "anthropic:claude-sonnet-5" if os.getenv("ANTHROPIC_API_KEY") else "google:gemini-flash-lite-latest"
path = data_processed_dir(f"synthetic_es_{N_CHATS}.jsonl")
if not path.exists():
    save_jsonl(await generate_chats(N_CHATS, GENERATOR, language="Spanish", rpm=RPM), path)
chats = load_jsonl(path)
print(f"{len(chats)}/{N_CHATS} chats (failed generations are skipped; delete the file to retry)", chats[0].meta)
print(chats[0].to_text())

In [ ]:
arc_results = pd.concat([await run_task(SENTIMENT_ARC, chats, m, rpm=RPM) for m in MODELS])
arc_scores = pd.concat([score(arc_results, chats, f).assign(field=f) for f in ["initial_sentiment", "final_sentiment"]])
arc_scores

In [ ]:
arc_results[["model", "id", "initial_sentiment", "final_sentiment", "final_sentiment_justification"]].head()

## Save
Copy the headline numbers into `docs/research.md`.

In [ ]:
out = reports_dir("benchmark")
out.mkdir(exist_ok=True)
pd.concat([tweet_scores.assign(task="message_sentiment"), arc_scores.assign(task="sentiment_arc")]).to_csv(out / "scores.csv", index=False)
pd.concat(tweet_results.values()).to_parquet(out / "tweet_results.parquet")
arc_results.to_parquet(out / "arc_results.parquet")